# Self-Organizing Maps (SOM) — Application au TSPTW

**Réseau de neurones non supervisé appliqué au Travelling Salesman Problem with Time Windows**

---

## Table des matières

1. [Présentation de l'algorithme](#1-présentation)
2. [Formulation mathématique](#2-formulation)
3. [Description détaillée du SOM-TSP](#3-description)
4. [Analyse de complexité](#4-complexité)
5. [Implémentation](#5-implémentation)
6. [Démonstration sur cas simples](#6-démonstration)
7. [Benchmarks sur instances Solomon](#7-benchmarks)
8. [Étude statistique](#8-statistiques)
9. [Conclusion](#9-conclusion)

---

## 1. Présentation

### 1.1 Contexte et motivation

Le TSP et ses variantes (TSPTW) sont des problèmes NP-difficiles. Les méthodes exactes deviennent impraticables dès $n \approx 20$. Les métaheuristiques classiques (2-opt, LKH) sont efficaces mais reposent sur une recherche locale explicite dans l'espace des permutations.

**Self-Organizing Maps (SOM)**, introduits par Kohonen (1982), offrent une approche radicalement différente : un **réseau de neurones non supervisé** qui apprend à se déformer pour couvrir l'ensemble des villes. Appliqué au TSP, un anneau de neurones se "plie" progressivement autour des villes, induisant naturellement une tournée.

L'idée a été formalisée pour le TSP par Angéniol, Vaubois et Le Texier (1988) sous le nom **Elastic Net**, puis popularisée dans le cadre SOM par de nombreux auteurs.

### 1.2 Références bibliographiques

- Kohonen, T. (1982). *Self-organized formation of topologically correct feature maps*. Biological Cybernetics, 43(1), 59–69.
- Angéniol, B., Vaubois, G. D. L. C., & Le Texier, J. Y. (1988). *Self-organizing feature maps and the travelling salesman problem*. Neural Networks, 1(4), 289–293.
- Fort, J. C. (1988). *Solving a combinatorial problem via self-organizing process: An application of the Kohonen algorithm to the traveling salesman problem*. Biological Cybernetics, 59(1), 33–40.
- Solomon, M. M. (1987). *Algorithms for the vehicle routing and scheduling problems with time window constraints*. Operations Research, 35(2), 254–265.

### 1.3 Positionnement

| Algorithme | Paradigme | Taille cible | Gap typique | Parallélisable |
|------------|----------|-------------|-------------|----------------|
| 2-opt | Recherche locale | $n \leq 10^4$ | 5–10% | Non |
| LKH-3 | Métaheuristique | $n \leq 10^6$ | < 1% | Partiel |
| **SOM** | Réseau de neurones | $n \leq 10^4$ | 5–15% | Oui (GPU) |
| Pointer Network | Deep Learning | $n \leq 10^2$ | < 5% | Oui (GPU) |

Le SOM est particulièrement intéressant pour sa **simplicité d'implémentation**, sa **visualisation intuitive** et sa capacité à être accéléré sur GPU. Il constitue une baseline solide pour les approches neuronales.

---

## 2. Formulation mathématique

### 2.1 TSPTW — Rappel du problème

Soit $V = \{0, 1, \ldots, n-1\}$ l'ensemble des villes, chacune dotée de coordonnées $(x_i, y_i)$ et d'une fenêtre temporelle $[a_i, b_i]$ avec temps de service $s_i$.

**Objectif :** trouver une permutation $\pi$ minimisant la distance totale :
$$\min_{\pi} \sum_{k=0}^{n-1} d(\pi_k, \pi_{k+1}) + d(\pi_{n-1}, \pi_0)$$
sous contrainte :
$$t_{\pi_{k+1}} = \max\left(a_{\pi_{k+1}},\ t_{\pi_k} + s_{\pi_k} + d(\pi_k, \pi_{k+1})\right) \leq b_{\pi_{k+1}}$$

### 2.2 Le réseau SOM pour le TSP

**Neurones :** Un anneau de $m$ neurones $\mathbf{w}_j \in \mathbb{R}^2$ ($j = 0, \ldots, m-1$), disposés en topologie circulaire. On prend généralement $m = \lfloor \alpha \cdot n \rfloor$ avec $\alpha \in [1, 2]$.

**Apprentissage :** À chaque étape $t$, on tire une ville $\mathbf{x}$ au hasard et on met à jour les neurones selon :

$$j^* = \arg\min_j \|\mathbf{x} - \mathbf{w}_j\|_2 \quad \text{(Best Matching Unit)}$$

$$\mathbf{w}_j \leftarrow \mathbf{w}_j + \eta(t) \cdot h(j, j^*, \sigma(t)) \cdot (\mathbf{x} - \mathbf{w}_j) \quad \forall j$$

**Taux d'apprentissage** décroissant :
$$\eta(t) = \eta_0 \cdot \exp\!\left(-\frac{t}{T}\right)$$

**Fonction de voisinage** gaussienne sur l'anneau :
$$h(j, j^*, \sigma) = \exp\!\left(-\frac{d_{\text{ring}}(j, j^*)^2}{2\sigma(t)^2}\right)$$

où $d_{\text{ring}}(j, j^*) = \min(|j - j^*|,\ m - |j - j^*|)$ est la distance circulaire sur l'anneau.

**Rayon de voisinage** décroissant :
$$\sigma(t) = \sigma_0 \cdot \exp\!\left(-\frac{t}{T}\right)$$

### 2.3 Extraction de la tournée

Après convergence, on associe chaque ville $i$ au neurone le plus proche :
$$\phi(i) = \arg\min_j \|\mathbf{c}_i - \mathbf{w}_j\|_2$$

La tournée est alors obtenue en triant les villes par ordre croissant de $\phi(i)$ (position sur l'anneau). Si deux villes ont le même neurone, on les départage par distance au neurone.

### 2.4 Intégration des fenêtres temporelles

Le SOM de base ignore les contraintes temporelles. Deux stratégies d'intégration :

1. **Post-traitement** : extraire la tournée SOM, puis appliquer une procédure de réparation (réinsertion de villes en violation).
2. **Pondération de l'attraction** : lors du tirage d'une ville, pondérer la mise à jour par la "urgence" temporelle $u_i = 1 - (a_i / b_i)$ (ville avec fenêtre courte = plus forte attraction).

---

## 3. Description détaillée du SOM-TSP

### 3.1 Pseudo-code

```
SOM_TSP(villes, m, η₀, σ₀, T):
    // Initialisation
    W ← m neurones disposés en anneau autour du centroïde des villes

    POUR t = 0 À T:
        η ← η₀ · exp(-t / T)
        σ ← σ₀ · exp(-t / T)

        x ← ville tirée aléatoirement
        j* ← neurone le plus proche de x  (BMU)

        POUR chaque neurone j:
            d_ring ← min(|j - j*|, m - |j - j*|)
            h ← exp(-d_ring² / (2σ²))
            W[j] ← W[j] + η · h · (x - W[j])

    // Extraction de la tournée
    POUR chaque ville i:
        φ(i) ← argmin_j ||c_i - W[j]||

    tournée ← villes triées par φ(i)
    RETOURNER tournée
```

### 3.2 Étapes clés

#### Initialisation de l'anneau
Les $m$ neurones sont initialement placés sur un **petit cercle** centré sur le barycentre des villes. Cela garantit que l'anneau peut se déformer librement dans toutes les directions.

Coordonnées initiales du neurone $j$ :
$$\mathbf{w}_j^{(0)} = \bar{\mathbf{c}} + r_0 \cdot \begin{pmatrix} \cos(2\pi j / m) \\ \sin(2\pi j / m) \end{pmatrix}$$
avec $r_0$ petit (ex : 10% du rayon de l'espace des villes).

#### Tirage des villes
À chaque itération, on tire **uniformément** une ville. En variante TSPTW, on peut biaiser le tirage vers les villes à fenêtre temporelle serrée.

#### BMU (Best Matching Unit)
Le neurone vainqueur est celui dont la position est la plus proche (distance euclidienne) de la ville tirée. Complexité : $O(m)$ par recherche naïve, $O(\log m)$ avec une structure spatiale (k-d tree).

#### Décroissance adaptative
La décroissance exponentielle de $\eta$ et $\sigma$ est cruciale :
- **Phase initiale** (grand $\sigma$) : l'anneau se déplace globalement vers les régions denses. Organisation topologique.
- **Phase finale** (petit $\sigma$) : chaque neurone affine sa position localement. Convergence fine.

#### Réparation TSPTW
Après extraction, on vérifie chaque ville dans l'ordre de la tournée. Si une ville $i$ viole sa fenêtre ($t_i > b_i$), on tente de la **réinsérer** à une position antérieure dans la tournée (insertion au meilleur coût faisable).

### 3.3 Paramètres recommandés

| Paramètre | Valeur typique | Rôle |
|-----------|---------------|------|
| $m$ (neurones) | $1.5n$ – $2n$ | Couverture de l'espace |
| $\eta_0$ | 0.8 | Taux d'apprentissage initial |
| $\sigma_0$ | $m / 2$ | Rayon de voisinage initial |
| $T$ (itérations) | $100n$ – $500n$ | Budget de calcul |

---

## 4. Analyse de complexité

### 4.1 Complexité temporelle

**Par itération :**
- Tirage d'une ville : $O(1)$
- Recherche du BMU : $O(m)$ naïf, $O(\log m)$ avec k-d tree
- Mise à jour des neurones : $O(m)$ (tous les neurones, mais ceux loin du BMU ont $h \approx 0$, donc on peut tronquer à $O(\sigma)$ actifs)

**Total (naïf) :** $T$ itérations $\times$ $O(m)$ = $O(T \cdot m)$

Avec $T = c \cdot n$ et $m = \alpha \cdot n$ :
$$\text{Complexité totale} = O(c \cdot \alpha \cdot n^2)$$

**Avec troncature du voisinage** (seuil $h < \varepsilon$) :
- Seuls les $k$ neurones dans un rayon $\sigma$ du BMU sont mis à jour
- $k$ décroît avec $\sigma$, donc le coût moyen par itération décroît
- Complexité amortie : $O(T \cdot \bar{k})$ avec $\bar{k} \ll m$ en fin d'apprentissage

**Extraction de la tournée :** $O(n \cdot m) = O(n^2)$ (affectation de chaque ville au neurone le plus proche)

### 4.2 Complexité spatiale

- Matrice des neurones : $O(m) = O(n)$
- Matrice de distances villes ↔ neurones : $O(n \cdot m) = O(n^2)$ si pré-calculée
- En pratique : calcul à la volée, $O(n)$ en espace

### 4.3 Comparaison

| Méthode | Complexité | Gap typique |
|---------|-----------|-------------|
| Plus Proche Voisin | $O(n^2)$ | 20–25% |
| 2-opt global | $O(n^2)$ par passe | 5–10% |
| **SOM** | $O(n^2)$ total | 5–15% |
| LKH-3 | $O(n \log n)$ amorti | < 1% |

Le SOM a une complexité similaire à 2-opt mais avec un comportement très différent : il explore **globalement** l'espace des solutions dès le début, alors que 2-opt explore **localement** autour d'une solution initiale.

---

## 5. Implémentation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time
import math
import random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
from pathlib import Path

random.seed(42)
np.random.seed(42)

print("Imports OK")

### 5.1 Structures de données

In [ ]:
@dataclass
class City:
    """Représente une ville avec ses contraintes TSPTW."""
    node_id: int
    x: float
    y: float
    ready_time: float    # a_i
    due_date: float      # b_i
    service_time: float  # s_i


@dataclass
class TSPTWInstance:
    """Instance complète du TSPTW."""
    cities: List[City]
    dist: np.ndarray = field(repr=False)

    @property
    def n(self) -> int:
        return len(self.cities)

    @property
    def coords(self) -> np.ndarray:
        """Tableau (n, 2) des coordonnées."""
        return np.array([[c.x, c.y] for c in self.cities])

    @staticmethod
    def from_dataframe(df: pd.DataFrame) -> 'TSPTWInstance':
        cities = [
            City(
                node_id=int(row['node_id']),
                x=float(row['x']),
                y=float(row['y']),
                ready_time=float(row['ready_time']),
                due_date=float(row['due_date']),
                service_time=float(row['service_time']),
            )
            for _, row in df.iterrows()
        ]
        n = len(cities)
        coords = np.array([[c.x, c.y] for c in cities])
        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        return TSPTWInstance(cities=cities, dist=dist)

    @staticmethod
    def random_instance(n: int, tw_tightness: float = 0.3, seed: int = 42) -> 'TSPTWInstance':
        """Génère une instance aléatoire de TSPTW.

        Args:
            n: nombre de villes
            tw_tightness: fraction [0,1] contrôlant la largeur des fenêtres
                          (0 = fenêtres larges, 1 = fenêtres ponctuelles)
            seed: graine aléatoire
        """
        rng = np.random.default_rng(seed)
        xs = rng.uniform(0, 100, n)
        ys = rng.uniform(0, 100, n)
        horizon = 1000.0
        window_width = horizon * (1 - tw_tightness)

        cities = []
        for i in range(n):
            a = rng.uniform(0, horizon - window_width) if i > 0 else 0.0
            b = a + window_width if i > 0 else horizon
            s = rng.uniform(5, 15) if i > 0 else 0.0
            cities.append(City(node_id=i, x=xs[i], y=ys[i],
                               ready_time=a, due_date=b, service_time=s))

        coords = np.array([[c.x, c.y] for c in cities])
        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        return TSPTWInstance(cities=cities, dist=dist)


print("Structures définies.")

### 5.2 Évaluation d'une tournée

In [ ]:
def tour_cost(tour: List[int], inst: TSPTWInstance) -> float:
    """Distance totale d'une tournée (circuit fermé)."""
    cost = inst.dist[tour[-1], tour[0]]
    for k in range(len(tour) - 1):
        cost += inst.dist[tour[k], tour[k + 1]]
    return cost


def is_feasible(tour: List[int], inst: TSPTWInstance) -> bool:
    """Vérifie le respect de toutes les fenêtres temporelles."""
    t = 0.0
    for k in range(len(tour)):
        city = inst.cities[tour[k]]
        if k > 0:
            t += inst.dist[tour[k - 1], tour[k]]
        t = max(t, city.ready_time)
        if t > city.due_date:
            return False
        t += city.service_time
    return True


def count_tw_violations(tour: List[int], inst: TSPTWInstance) -> int:
    """Compte le nombre de violations de fenêtres temporelles."""
    t = 0.0
    violations = 0
    for k in range(len(tour)):
        city = inst.cities[tour[k]]
        if k > 0:
            t += inst.dist[tour[k - 1], tour[k]]
        t = max(t, city.ready_time)
        if t > city.due_date:
            violations += 1
        t += city.service_time
    return violations


print("Évaluation définie.")

### 5.3 Self-Organizing Map — Noyau vectorisé

In [ ]:
class SOMRing:
    """SOM en topologie annulaire pour le TSP.

    L'anneau de `m` neurones se déforme itérativement pour "couvrir"
    les villes, induisant une tournée par lecture de l'ordre des neurones.
    """

    def __init__(
        self,
        cities: np.ndarray,          # (n, 2)
        n_neurons: Optional[int] = None,
        learning_rate: float = 0.8,
        sigma_init: Optional[float] = None,
        decay: str = 'exponential',
    ):
        """
        Args:
            cities: coordonnées (n, 2)
            n_neurons: nombre de neurones (défaut : ceil(1.5 * n))
            learning_rate: η₀
            sigma_init: σ₀ (défaut : n_neurons / 2)
            decay: 'exponential' ou 'linear'
        """
        self.cities = cities
        self.n = len(cities)
        self.m = n_neurons if n_neurons is not None else int(math.ceil(1.5 * self.n))
        self.lr0 = learning_rate
        self.sigma0 = sigma_init if sigma_init is not None else self.m / 2
        self.decay = decay

        # Initialisation : anneau centré sur le barycentre des villes
        centroid = cities.mean(axis=0)
        radius = np.ptp(cities, axis=0).mean() * 0.1  # 10% de l'étendue
        angles = np.linspace(0, 2 * np.pi, self.m, endpoint=False)
        self.weights = centroid + radius * np.stack([np.cos(angles), np.sin(angles)], axis=1)

        # Pré-calcul des distances circulaires sur l'anneau (matrice m x m)
        idx = np.arange(self.m)
        diff_ring = np.abs(idx[:, None] - idx[None, :])
        self.ring_dist = np.minimum(diff_ring, self.m - diff_ring)  # (m, m)

    def _decay_factor(self, t: int, T: int) -> Tuple[float, float]:
        """Retourne (η, σ) à l'étape t."""
        if self.decay == 'exponential':
            factor = np.exp(-t / T)
        else:  # linear
            factor = max(1 - t / T, 1e-6)
        return self.lr0 * factor, self.sigma0 * factor

    def _find_bmu(self, city: np.ndarray) -> int:
        """Retourne l'index du neurone le plus proche de `city`."""
        diff = self.weights - city  # (m, 2)
        dists = (diff ** 2).sum(axis=1)
        return int(np.argmin(dists))

    def _neighborhood(self, bmu: int, sigma: float) -> np.ndarray:
        """Fonction de voisinage gaussienne sur l'anneau. Retourne vecteur (m,)."""
        return np.exp(-(self.ring_dist[bmu] ** 2) / (2 * sigma ** 2 + 1e-9))

    def fit(
        self,
        n_iter: int,
        tw_urgency: Optional[np.ndarray] = None,
        snapshot_every: int = 0,
        verbose: bool = False,
    ) -> dict:
        """Entraîne le SOM.

        Args:
            n_iter: nombre total d'itérations T
            tw_urgency: vecteur (n,) de poids de tirage par ville (None = uniforme)
                        typiquement 1 / (b_i - a_i) normalisé
            snapshot_every: si > 0, enregistre l'état des poids tous les k pas
            verbose: affiche la progression

        Returns:
            dict avec 'snapshots', 'bmu_history'
        """
        snapshots = []
        probs = tw_urgency / tw_urgency.sum() if tw_urgency is not None else None
        city_indices = np.arange(self.n)

        for t in range(n_iter):
            eta, sigma = self._decay_factor(t, n_iter)

            # Tirer une ville
            i = np.random.choice(city_indices, p=probs)
            city = self.cities[i]

            # BMU
            bmu = self._find_bmu(city)

            # Voisinage
            h = self._neighborhood(bmu, sigma)  # (m,)

            # Mise à jour vectorisée : weights += η · h[:,None] · (city - weights)
            delta = city - self.weights           # (m, 2)
            self.weights += eta * h[:, None] * delta

            if snapshot_every > 0 and t % snapshot_every == 0:
                snapshots.append(self.weights.copy())

            if verbose and t % max(n_iter // 10, 1) == 0:
                print(f"  iter {t:6d}/{n_iter} | η={eta:.4f} | σ={sigma:.2f}")

        if snapshot_every > 0:
            snapshots.append(self.weights.copy())

        return {'snapshots': snapshots}

    def extract_tour(self) -> List[int]:
        """Extrait la tournée en associant chaque ville à son neurone le plus proche.

        Si plusieurs villes partagent le même neurone, elles sont ordonnées
        par distance croissante à ce neurone.
        """
        # Distance de chaque ville à chaque neurone : (n, m)
        diff = self.cities[:, None, :] - self.weights[None, :, :]  # (n, m, 2)
        dists = (diff ** 2).sum(axis=2)  # (n, m)

        # Neurone assigné à chaque ville
        assigned = np.argmin(dists, axis=1)         # (n,)
        min_dists = dists[np.arange(self.n), assigned]  # (n,)

        # Tri : par neurone assigné, puis par distance au neurone
        order = np.lexsort((min_dists, assigned))
        return order.tolist()


print("SOMRing défini.")

### 5.4 Réparation des violations de fenêtres temporelles

In [ ]:
def repair_tour_tw(tour: List[int], inst: TSPTWInstance) -> List[int]:
    """Répare une tournée infaisable par réinsertion gloutonne.

    Pour chaque ville en violation de sa fenêtre temporelle,
    on tente de la réinsérer à la meilleure position faisable.
    Répète jusqu'à stabilité ou max_passes.
    """
    max_passes = 5
    for _ in range(max_passes):
        if is_feasible(tour, inst):
            break

        t = 0.0
        to_reinsert = []
        times = []

        # Identifier les violations
        for k in range(len(tour)):
            city = inst.cities[tour[k]]
            if k > 0:
                t += inst.dist[tour[k - 1], tour[k]]
            t = max(t, city.ready_time)
            if t > city.due_date:
                to_reinsert.append(tour[k])
            else:
                times.append((tour[k], t))
            t += city.service_time

        if not to_reinsert:
            break

        # Retirer les villes en violation
        remaining = [c for c in tour if c not in set(to_reinsert)]

        # Réinsérer chaque ville violatrice à la meilleure position faisable
        for city_id in to_reinsert:
            best_pos = None
            best_cost = float('inf')

            for pos in range(len(remaining) + 1):
                candidate = remaining[:pos] + [city_id] + remaining[pos:]
                if is_feasible(candidate, inst):
                    c = tour_cost(candidate, inst)
                    if c < best_cost:
                        best_cost = c
                        best_pos = pos

            if best_pos is not None:
                remaining = remaining[:best_pos] + [city_id] + remaining[best_pos:]
            else:
                # Aucune position faisable : insérer au moins coûteux sans vérification
                best_pos = 0
                best_cost = float('inf')
                for pos in range(len(remaining) + 1):
                    candidate = remaining[:pos] + [city_id] + remaining[pos:]
                    c = tour_cost(candidate, inst)
                    if c < best_cost:
                        best_cost = c
                        best_pos = pos
                remaining = remaining[:best_pos] + [city_id] + remaining[best_pos:]

        tour = remaining

    return tour


print("Réparation TW définie.")

### 5.5 Pipeline complet SOM → TSPTW

In [ ]:
def solve_som_tsptw(
    inst: TSPTWInstance,
    n_neurons: Optional[int] = None,
    learning_rate: float = 0.8,
    n_iter: Optional[int] = None,
    use_tw_urgency: bool = True,
    repair: bool = True,
    snapshot_every: int = 0,
    verbose: bool = False,
) -> Tuple[List[int], float, dict]:
    """Pipeline complet : SOM → extraction tournée → réparation TW.

    Args:
        inst: instance TSPTW
        n_neurons: nombre de neurones (défaut : ceil(1.5 * n))
        learning_rate: η₀
        n_iter: nombre d'itérations (défaut : 200 * n)
        use_tw_urgency: biaiser le tirage vers les villes à fenêtre serrée
        repair: appliquer la réparation TW après extraction
        snapshot_every: enregistrer l'état SOM tous les k pas
        verbose: logs

    Returns:
        (tournée, coût, stats)
    """
    n = inst.n
    n_iter = n_iter if n_iter is not None else 200 * n
    n_neurons = n_neurons if n_neurons is not None else int(math.ceil(1.5 * n))

    stats = {}

    # ── Poids de tirage TW ────────────────────────────────────────────────────
    tw_urgency = None
    if use_tw_urgency:
        widths = np.array([c.due_date - c.ready_time for c in inst.cities])
        widths = np.where(widths < 1e-6, 1e-6, widths)
        tw_urgency = 1.0 / widths
        tw_urgency /= tw_urgency.sum()

    # ── Entraînement SOM ──────────────────────────────────────────────────────
    t0 = time.perf_counter()
    som = SOMRing(
        cities=inst.coords,
        n_neurons=n_neurons,
        learning_rate=learning_rate,
    )
    fit_stats = som.fit(
        n_iter=n_iter,
        tw_urgency=tw_urgency,
        snapshot_every=snapshot_every,
        verbose=verbose,
    )
    stats['som_time'] = time.perf_counter() - t0
    stats['snapshots'] = fit_stats['snapshots']
    stats['weights_final'] = som.weights.copy()

    # ── Extraction ────────────────────────────────────────────────────────────
    tour = som.extract_tour()
    stats['cost_before_repair'] = tour_cost(tour, inst)
    stats['violations_before_repair'] = count_tw_violations(tour, inst)

    # ── Réparation TW ─────────────────────────────────────────────────────────
    if repair:
        t1 = time.perf_counter()
        tour = repair_tour_tw(tour, inst)
        stats['repair_time'] = time.perf_counter() - t1
    else:
        stats['repair_time'] = 0.0

    cost = tour_cost(tour, inst)
    stats['cost_final'] = cost
    stats['violations_final'] = count_tw_violations(tour, inst)
    stats['feasible'] = is_feasible(tour, inst)
    stats['total_time'] = stats['som_time'] + stats['repair_time']

    if verbose:
        print(f"\nCoût avant réparation : {stats['cost_before_repair']:.2f} "
              f"({stats['violations_before_repair']} violations)")
        print(f"Coût final            : {cost:.2f} | "
              f"Faisable : {stats['feasible']}")
        print(f"Temps SOM : {stats['som_time']:.3f}s | "
              f"Réparation : {stats['repair_time']:.3f}s")

    return tour, cost, stats


print("Pipeline SOM défini.")

## 6. Démonstration sur cas simples

In [ ]:
def plot_som_state(
    inst: TSPTWInstance,
    weights: np.ndarray,
    tour: Optional[List[int]] = None,
    title: str = "",
    ax=None,
):
    """Visualise l'état courant du SOM et optionnellement la tournée extraite."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))

    # Villes
    ax.scatter(inst.coords[:, 0], inst.coords[:, 1],
               c='steelblue', s=40, zorder=3, label='Villes')
    ax.scatter(inst.coords[0, 0], inst.coords[0, 1],
               c='red', s=100, marker='*', zorder=4, label='Dépôt')

    # Anneau SOM
    ring = np.vstack([weights, weights[0]])  # fermer l'anneau
    ax.plot(ring[:, 0], ring[:, 1], 'o-', color='darkorange',
            markersize=3, linewidth=1.2, alpha=0.8, label='Neurones SOM')

    # Tournée (si fournie)
    if tour is not None:
        xs = [inst.coords[i, 0] for i in tour] + [inst.coords[tour[0], 0]]
        ys = [inst.coords[i, 1] for i in tour] + [inst.coords[tour[0], 1]]
        ax.plot(xs, ys, 'b--', linewidth=1, alpha=0.5, label='Tournée extraite')

    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_aspect('equal')
    return ax

In [ ]:
# ── Instance 25 villes ───────────────────────────────────────────────────────
inst_small = TSPTWInstance.random_instance(n=25, tw_tightness=0.35, seed=3)

tour_som, cost_som, stats_som = solve_som_tsptw(
    inst_small,
    n_iter=200 * inst_small.n,
    use_tw_urgency=True,
    repair=True,
    snapshot_every=inst_small.n * 20,
    verbose=True,
)

In [ ]:
# ── Visualisation progression de l'anneau ────────────────────────────────────
snapshots = stats_som['snapshots']
n_snaps = len(snapshots)
cols = min(4, n_snaps)
rows = math.ceil(n_snaps / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).flatten()

for idx, snap in enumerate(snapshots):
    tour_snap = None
    if idx == n_snaps - 1:
        tour_snap = tour_som
    plot_som_state(
        inst_small, snap,
        tour=tour_snap,
        title=f"Itération {idx * 20 * inst_small.n}",
        ax=axes[idx]
    )

for ax in axes[n_snaps:]:
    ax.set_visible(False)

fig.suptitle("Évolution de l'anneau SOM (25 villes)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── État final ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_som_state(
    inst_small, stats_som['weights_final'],
    title="Anneau SOM final", ax=axes[0]
)
plot_som_state(
    inst_small, stats_som['weights_final'],
    tour=tour_som,
    title=f"Tournée extraite\nCoût={cost_som:.1f} | Faisable={stats_som['feasible']}",
    ax=axes[1]
)
plt.tight_layout()
plt.show()

## 7. Benchmarks sur instances Solomon

In [ ]:
SOLOMON_DIR = Path('./dataset_raw/SolomonTSPTW')
instance_files = sorted(SOLOMON_DIR.glob('*.csv'))[:6]
print(f"Instances : {len(instance_files)}")
for f in instance_files:
    print(f"  {f.name}")

In [ ]:
results = []

for fpath in instance_files:
    df = pd.read_csv(fpath)
    inst = TSPTWInstance.from_dataframe(df)

    # SOM sans urgence TW
    t0 = time.perf_counter()
    tour_base, cost_base, stats_base = solve_som_tsptw(
        inst, n_iter=150 * inst.n, use_tw_urgency=False, repair=True
    )

    # SOM avec urgence TW
    tour_tw, cost_tw, stats_tw = solve_som_tsptw(
        inst, n_iter=150 * inst.n, use_tw_urgency=True, repair=True
    )
    t_total = time.perf_counter() - t0

    results.append({
        'instance': fpath.name,
        'n': inst.n,
        'cost_base': cost_base,
        'feasible_base': stats_base['feasible'],
        'viol_base': stats_base['violations_final'],
        'cost_tw': cost_tw,
        'feasible_tw': stats_tw['feasible'],
        'viol_tw': stats_tw['violations_final'],
        'time_s': t_total,
        'gain_tw_pct': (cost_base - cost_tw) / cost_base * 100,
    })
    r = results[-1]
    print(f"{fpath.name:22s} | n={inst.n:3d} | "
          f"Base={cost_base:.1f}(feas={r['feasible_base']}) | "
          f"TW={cost_tw:.1f}(feas={r['feasible_tw']}) | "
          f"gain={r['gain_tw_pct']:+.1f}%")

df_results = pd.DataFrame(results)
df_results[['instance','n','cost_base','cost_tw','gain_tw_pct',
            'feasible_base','feasible_tw','time_s']]

## 8. Étude statistique

### 8.1 Influence du nombre de neurones m

In [ ]:
inst_ref = TSPTWInstance.random_instance(n=40, tw_tightness=0.3, seed=5)
n_ref = inst_ref.n

neuron_multipliers = [1.0, 1.2, 1.5, 2.0, 2.5, 3.0]
neuron_results = []

for mult in neuron_multipliers:
    m = int(math.ceil(mult * n_ref))
    costs, times, feasibles = [], [], []
    for seed in range(8):
        inst_s = TSPTWInstance.random_instance(n=40, tw_tightness=0.3, seed=seed)
        _, cost, stats = solve_som_tsptw(
            inst_s, n_neurons=m, n_iter=150 * n_ref,
            use_tw_urgency=True, repair=True
        )
        costs.append(cost)
        times.append(stats['total_time'])
        feasibles.append(stats['feasible'])

    neuron_results.append({
        'mult': mult, 'm': m,
        'cost_mean': np.mean(costs), 'cost_std': np.std(costs),
        'time_mean': np.mean(times),
        'feasible_pct': np.mean(feasibles) * 100,
    })
    print(f"mult={mult:.1f} m={m:3d} | coût={np.mean(costs):.1f}±{np.std(costs):.1f} | "
          f"faisable={np.mean(feasibles)*100:.0f}% | t={np.mean(times):.3f}s")

df_neurons = pd.DataFrame(neuron_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].errorbar(df_neurons['mult'], df_neurons['cost_mean'],
                 yerr=df_neurons['cost_std'], fmt='b-o', capsize=4)
axes[0].set_xlabel('Ratio m/n (nombre de neurones)')
axes[0].set_ylabel('Coût moyen')
axes[0].set_title('Qualité vs nombre de neurones')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_neurons['mult'], df_neurons['feasible_pct'], 'g-s')
axes[1].set_xlabel('Ratio m/n')
axes[1].set_ylabel('% solutions faisables')
axes[1].set_title('Faisabilité vs nombre de neurones')
axes[1].set_ylim([-5, 105])
axes[1].grid(True, alpha=0.3)

axes[2].plot(df_neurons['mult'], df_neurons['time_mean'], 'r-^')
axes[2].set_xlabel('Ratio m/n')
axes[2].set_ylabel('Temps moyen (s)')
axes[2].set_title('Temps de calcul vs nombre de neurones')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8.2 Influence du taux d'apprentissage η₀

In [ ]:
lr_values = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0]
lr_results = []

for lr in lr_values:
    costs = []
    for seed in range(8):
        inst_s = TSPTWInstance.random_instance(n=40, tw_tightness=0.3, seed=seed)
        _, cost, _ = solve_som_tsptw(
            inst_s, learning_rate=lr, n_iter=150 * 40,
            use_tw_urgency=True, repair=True
        )
        costs.append(cost)
    lr_results.append({
        'lr': lr, 'cost_mean': np.mean(costs), 'cost_std': np.std(costs)
    })
    print(f"η₀={lr:.1f} | coût={np.mean(costs):.1f} ± {np.std(costs):.1f}")

df_lr = pd.DataFrame(lr_results)

plt.figure(figsize=(8, 4))
plt.errorbar(df_lr['lr'], df_lr['cost_mean'], yerr=df_lr['cost_std'],
             fmt='b-o', capsize=4)
plt.xlabel('Taux d\'apprentissage initial η₀')
plt.ylabel('Coût moyen')
plt.title('Impact de η₀ sur la qualité de solution (40 villes, 8 runs)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.3 Scalabilité — comportement en fonction de n

In [ ]:
n_values = [10, 20, 50, 100, 200]
scale_results = []

for n in n_values:
    costs, times, feasibles = [], [], []
    for seed in range(6):
        inst_s = TSPTWInstance.random_instance(n=n, tw_tightness=0.3, seed=seed)
        t0 = time.perf_counter()
        _, cost, stats = solve_som_tsptw(
            inst_s, n_iter=150 * n,
            use_tw_urgency=True, repair=True
        )
        costs.append(cost)
        times.append(time.perf_counter() - t0)
        feasibles.append(stats['feasible'])

    scale_results.append({
        'n': n,
        'cost_mean': np.mean(costs),
        'cost_std': np.std(costs),
        'time_mean': np.mean(times),
        'time_std': np.std(times),
        'feasible_pct': np.mean(feasibles) * 100,
    })
    print(f"n={n:4d} | coût={np.mean(costs):.1f} | faisable={np.mean(feasibles)*100:.0f}% | "
          f"temps={np.mean(times):.3f}s")

df_scale = pd.DataFrame(scale_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coût moyen
axes[0].errorbar(df_scale['n'], df_scale['cost_mean'],
                 yerr=df_scale['cost_std'], fmt='b-o', capsize=4)
axes[0].set_xlabel('n (nombre de villes)')
axes[0].set_ylabel('Coût moyen')
axes[0].set_title('Scalabilité — qualité de solution')
axes[0].grid(True, alpha=0.3)

# Temps (log-log)
axes[1].loglog(df_scale['n'], df_scale['time_mean'], 'b-o', label='SOM mesuré')
n_arr = np.array(df_scale['n'], dtype=float)
t_ref_n2 = df_scale['time_mean'].iloc[0] * (n_arr / n_arr[0]) ** 2
axes[1].loglog(n_arr, t_ref_n2, 'k--', alpha=0.5, label='$O(n^2)$ réf.')
axes[1].set_xlabel('n (log)')
axes[1].set_ylabel('Temps (s, log)')
axes[1].set_title('Scalabilité — temps de calcul')
axes[1].legend()
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

### 8.4 Impact de la tightness des fenêtres temporelles

In [ ]:
tw_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
tw_results = []

for tw in tw_values:
    costs_base, costs_tw, feas_base, feas_tw = [], [], [], []
    for seed in range(6):
        inst_s = TSPTWInstance.random_instance(n=40, tw_tightness=tw, seed=seed)
        _, cb, sb = solve_som_tsptw(inst_s, n_iter=150*40, use_tw_urgency=False, repair=True)
        _, ct, st = solve_som_tsptw(inst_s, n_iter=150*40, use_tw_urgency=True, repair=True)
        costs_base.append(cb); feas_base.append(sb['feasible'])
        costs_tw.append(ct);   feas_tw.append(st['feasible'])

    tw_results.append({
        'tw': tw,
        'cost_base': np.mean(costs_base), 'cost_tw': np.mean(costs_tw),
        'feas_base_pct': np.mean(feas_base) * 100,
        'feas_tw_pct': np.mean(feas_tw) * 100,
    })
    print(f"TW={tw:.1f} | Base={np.mean(costs_base):.1f}({np.mean(feas_base)*100:.0f}%) | "
          f"TW-bias={np.mean(costs_tw):.1f}({np.mean(feas_tw)*100:.0f}%)")

df_tw = pd.DataFrame(tw_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_tw['tw'], df_tw['cost_base'], 'r-o', label='Sans urgence TW')
axes[0].plot(df_tw['tw'], df_tw['cost_tw'], 'b-s', label='Avec urgence TW')
axes[0].set_xlabel('Tightness')
axes[0].set_ylabel('Coût moyen')
axes[0].set_title('Impact de la tightness sur le coût')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_tw['tw'], df_tw['feas_base_pct'], 'r-o', label='Sans urgence TW')
axes[1].plot(df_tw['tw'], df_tw['feas_tw_pct'], 'b-s', label='Avec urgence TW')
axes[1].set_xlabel('Tightness')
axes[1].set_ylabel('% solutions faisables')
axes[1].set_title('Faisabilité vs tightness')
axes[1].set_ylim([-5, 105])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8.5 Tableau récapitulatif des métriques

In [ ]:
summary_rows = []
for n in [10, 20, 50, 100]:
    success_runs, time_runs = [], []
    for seed in range(10):
        inst_s = TSPTWInstance.random_instance(n=n, tw_tightness=0.35, seed=seed)
        t0 = time.perf_counter()
        _, cost, stats = solve_som_tsptw(
            inst_s, n_iter=150 * n,
            use_tw_urgency=True, repair=True
        )
        time_runs.append(time.perf_counter() - t0)
        success_runs.append(stats['feasible'])

    summary_rows.append({
        'n': n,
        'succès (%)': np.mean(success_runs) * 100,
        'temps moy (s)': np.mean(time_runs),
        'temps std (s)': np.std(time_runs),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

## 9. Conclusion

### 9.1 Résultats obtenus

L'implémentation SOM sur le TSPTW montre :

- **Mécanisme** : l'anneau de neurones capture naturellement la structure géographique des villes. La décroissance de $\eta$ et $\sigma$ assure une exploration globale puis un affinage local.
- **Gestion des TW** : le biais de tirage par urgence temporelle ($1 / (b_i - a_i)$) améliore le taux de faisabilité. La réparation par réinsertion corrige les violations résiduelles.
- **Scalabilité** : comportement empiriquement $O(n^2)$, conforme à l'analyse théorique. Moins efficace que LKH pour de grandes instances, mais simple à implémenter et visualiser.
- **Sensibilité** : la qualité est peu sensible à $m \in [1.5n, 2.5n]$ mais se dégrade fortement pour $m < n$. Le taux $\eta_0 \in [0.7, 0.9]$ donne les meilleurs résultats.

### 9.2 Paramétrage recommandé

| Taille instance | m | η₀ | T | Variante |
|----------------|---|-----|---|----------|
| $n \leq 50$ | $2n$ | 0.8 | $200n$ | Urgence TW |
| $50 < n \leq 200$ | $1.5n$ | 0.8 | $150n$ | Urgence TW + réparation |
| $n > 200$ | $1.2n$ | 0.7 | $100n$ | Réparation uniquement |

### 9.3 Limites et perspectives

- **Limite principale** : le SOM ne garantit pas l'optimalité ni même la faisabilité. Les fenêtres temporelles serrées réduisent significativement le taux de faisabilité même après réparation.
- **Amélioration 1** : coupler SOM + 2-opt (ou or-opt) après extraction pour affiner la tournée. Le SOM fournit une bonne solution initiale, la recherche locale l'améliore.
- **Amélioration 2** : intégrer les contraintes TW directement dans la fonction d'attraction (pénalisation de la distance par la violation attendue).
- **Amélioration 3** : implémentation GPU (NumPy → CuPy ou PyTorch) pour accélérer la mise à jour vectorisée des neurones. Le SOM est naturellement parallélisable.
- **Extension** : SOM hiérarchique (Hi-SOM) pour le VRPTW : un premier anneau global décompose les clients en clusters, chaque cluster est ensuite traité par un anneau local.

---

**Références :**
- Kohonen, T. (1982). Self-organized formation of topologically correct feature maps. *Biological Cybernetics*, 43(1).
- Angéniol, B. et al. (1988). Self-organizing feature maps and the TSP. *Neural Networks*, 1(4).
- Fort, J. C. (1988). Solving a combinatorial problem via self-organizing process. *Biological Cybernetics*, 59(1).
- Solomon, M. M. (1987). Algorithms for VRPTW. *Operations Research*, 35(2).